# Earth Engine Python API Colab Setup

This notebook demonstrates how to setup the Earth Engine Python API in Colab and provides several examples of how to print and visualize Earth Engine processed data.

## Import API and get credentials

The Earth Engine API is installed by default in Google Colaboratory so requires only importing and authenticating. These steps must be completed for each new Colab session, if you restart your Colab kernel, or if your Colab virtual machine is recycled due to inactivity.

### Import the API

Run the following cell to import the API into your session.

In [1]:
import ee

### Authenticate and initialize

Run the `ee.Authenticate` function to authenticate your access to Earth Engine servers and `ee.Initialize` to initialize it. Upon running the following cell you'll be asked to grant Earth Engine access to your Google account. Follow the instructions printed to the cell.

In [6]:
# Trigger the authentication flow.
ee.Authenticate()

# Initialize the library.
ee.Initialize(project='dekkal-04')



s1 = ee.ImageCollection('COPERNICUS/S1_GRD') \
    .filterBounds(ee.Geometry.Point([-17.45, 14.7])) \
    .filterDate('2024-01-01', '2024-01-31') \
    .filter(ee.Filter.eq('instrumentMode', 'IW')) \
    .select('VV')

print(s1.size().getInfo())

2


In [10]:
# CELLULE 1 — toujours exécuter en premier
import ee
ee.Initialize(project='dekkal-04')

# Zones géographiques Dëkkal
dakar = ee.Geometry.Rectangle([-17.6, 14.6, -17.1, 14.95])
pikine = ee.Geometry.Point([-17.39, 14.75])
keur_massar = ee.Geometry.Point([-17.33, 14.78])

print("GEE initialisé ✓")

GEE initialisé ✓


In [11]:

# CHIRPS journalier — saison pluies 2020-2024
chirps = (ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
    .filterBounds(dakar)
    .filterDate('2020-06-01', '2024-10-31')
    .select('precipitation'))

print(f"CHIRPS images : {chirps.size().getInfo()}")

# Global Flood Database
floods = (ee.ImageCollection('GLOBAL_FLOOD_DB/MODIS_EVENTS/V1')
    .filterBounds(dakar))

print(f"Événements inondation détectés : {floods.size().getInfo()}")

CHIRPS images : 1613
Événements inondation détectés : 7


In [13]:
# Voir les détails des 7 événements
import datetime

flood_list = floods.toList(10)

print("Événements d'inondation détectés sur Dakar :\n")
for i in range(7):
    img = ee.Image(flood_list.get(i))
    props = img.getInfo()['properties']
    
    # Convertir timestamp ms → date lisible
    ts_ms = props.get('system:time_start')
    date = datetime.datetime.fromtimestamp(ts_ms / 1000).strftime('%Y-%m-%d')
    flood_id = props.get('id', 'N/A')
    
    print(f"Événement {i+1} | ID: {flood_id} | Date: {date}")

Événements d'inondation détectés sur Dakar :

Événement 1 | ID: 2316 | Date: 2003-08-09
Événement 2 | ID: 2742 | Date: 2005-09-27
Événement 3 | ID: 3180 | Date: 2007-08-31
Événement 4 | ID: 3531 | Date: 2009-08-24
Événement 5 | ID: 3534 | Date: 2009-08-10
Événement 6 | ID: 3971 | Date: 2012-08-24
Événement 7 | ID: 3982 | Date: 2012-09-02


In [14]:
import pandas as pd

results = []

for i in range(7):
    img = ee.Image(flood_list.get(i))
    props = img.getInfo()['properties']
    ts_ms = props.get('system:time_start')
    date_str = datetime.datetime.fromtimestamp(ts_ms / 1000).strftime('%Y-%m-%d')
    flood_id = props.get('id')
    
    flood_date = ee.Date(ts_ms)
    
    # Précipitations cumulées J-1, J-3, J-7, J-30
    for window, label in [(1,'precip_1d'), (3,'precip_3d'), (7,'precip_7d'), (30,'precip_30d')]:
        cumul = (chirps
            .filterDate(flood_date.advance(-window, 'day'), flood_date)
            .sum()
            .reduceRegion(
                reducer=ee.Reducer.mean(),
                geometry=dakar,
                scale=5000
            ).getInfo())
        
        results.append({
            'flood_id': flood_id,
            'date': date_str,
            'window': label,
            'precip_mm': round(cumul.get('precipitation', 0), 1)
        })

df = pd.DataFrame(results)
df_pivot = df.pivot(index=['flood_id','date'], columns='window', values='precip_mm').reset_index()
print(df_pivot.to_string())

window  flood_id        date  precip_1d  precip_30d  precip_3d  precip_7d
0           2316  2003-08-09          0           0          0          0
1           2742  2005-09-27          0           0          0          0
2           3180  2007-08-31          0           0          0          0
3           3531  2009-08-24          0           0          0          0
4           3534  2009-08-10          0           0          0          0
5           3971  2012-08-24          0           0          0          0
6           3982  2012-09-02          0           0          0          0


In [15]:
import datetime
import pandas as pd

dakar = ee.Geometry.Rectangle([-17.6, 14.6, -17.1, 14.95])

# CHIRPS SANS filtre de date — toute la période
chirps_all = (ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
    .filterBounds(dakar)
    .select('precipitation'))

print(f"CHIRPS total : {chirps_all.size().getInfo()} images")

results = []

for i in range(7):
    img = ee.Image(flood_list.get(i))
    props = img.getInfo()['properties']
    ts_ms = props.get('system:time_start')
    date_str = datetime.datetime.fromtimestamp(ts_ms / 1000).strftime('%Y-%m-%d')
    flood_id = props.get('id')
    flood_date = ee.Date(ts_ms)

    for window, label in [(1,'precip_1d'), (3,'precip_3d'), (7,'precip_7d'), (30,'precip_30d')]:
        cumul = (chirps_all
            .filterDate(flood_date.advance(-window, 'day'), flood_date)
            .sum()
            .reduceRegion(
                reducer=ee.Reducer.mean(),
                geometry=dakar,
                scale=5000
            ).getInfo())

        results.append({
            'flood_id': flood_id,
            'date': date_str,
            'window': label,
            'precip_mm': round(cumul.get('precipitation', 0), 1)
        })

df = pd.DataFrame(results)
df_pivot = df.pivot(index=['flood_id','date'], columns='window', values='precip_mm').reset_index()
print(df_pivot.to_string())

CHIRPS total : 16526 images
window  flood_id        date  precip_1d  precip_30d  precip_3d  precip_7d
0           2316  2003-08-09        0.3        73.8        0.4        0.4
1           2742  2005-09-27       16.3       177.7       16.3       18.4
2           3180  2007-08-31        7.2        97.4       11.3       14.4
3           3531  2009-08-24       13.7       190.6       43.8       56.8
4           3534  2009-08-10        6.9       132.4       25.7       53.1
5           3971  2012-08-24        3.3       196.9        3.9       26.4
6           3982  2012-09-02       14.1       298.9       25.7      117.5


In [18]:
df_pivot['flood'] = 1
df_pivot['source'] = 'Global_Flood_DB'

# CSV en fallback
df_pivot.to_csv('flood_labels_dakar_2003_2012.csv', index=False)
print("Sauvegardé → flood_labels_dakar_2003_2012.csv")
print(df_pivot)

Sauvegardé → flood_labels_dakar_2003_2012.csv
window  flood_id        date  precip_1d  precip_30d  precip_3d  precip_7d  \
0           2316  2003-08-09        0.3        73.8        0.4        0.4   
1           2742  2005-09-27       16.3       177.7       16.3       18.4   
2           3180  2007-08-31        7.2        97.4       11.3       14.4   
3           3531  2009-08-24       13.7       190.6       43.8       56.8   
4           3534  2009-08-10        6.9       132.4       25.7       53.1   
5           3971  2012-08-24        3.3       196.9        3.9       26.4   
6           3982  2012-09-02       14.1       298.9       25.7      117.5   

window  flood           source  
0           1  Global_Flood_DB  
1           1  Global_Flood_DB  
2           1  Global_Flood_DB  
3           1  Global_Flood_DB  
4           1  Global_Flood_DB  
5           1  Global_Flood_DB  
6           1  Global_Flood_DB  


In [19]:
# SOURCE 1 — Global Flood DB (2000-2018, MODIS 500m)
floods_modis = (ee.ImageCollection('GLOBAL_FLOOD_DB/MODIS_EVENTS/V1')
    .filterBounds(dakar))
print(f"MODIS Flood DB : {floods_modis.size().getInfo()} événements")

# SOURCE 2 — JRC Surface Water (1984-2023, Landsat 30m)
# Détecte les pixels qui sont devenus eau de façon saisonnière
jrc = ee.Image('JRC/GSW1_4/GlobalSurfaceWater')
flood_extent = jrc.select('seasonal_water').clip(dakar)
print("JRC Surface Water chargé ✓")

# SOURCE 3 — Sentinel-1 SAR (2014-présent, 10m)
# Détection directe eau par backscatter — le plus précis
s1_floods = (ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterBounds(dakar)
    .filterDate('2014-07-01', '2024-10-31')
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    .select('VV'))
print(f"Sentinel-1 images : {s1_floods.size().getInfo()}")

MODIS Flood DB : 7 événements
JRC Surface Water chargé ✓
Sentinel-1 images : 525


In [ ]:
# Correction : ASCENDING au lieu de DESCENDING
WATER_THRESHOLD = -15

s1_flood_event = (ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterBounds(dakar)
    .filterDate('2020-08-01', '2020-09-30')
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    .filter(ee.Filter.eq('orbitProperties_pass', 'ASCENDING'))
    .select('VV')
    .mean())

water_mask = s1_flood_event.lt(WATER_THRESHOLD)

flood_area = water_mask.multiply(ee.Image.pixelArea()) \
    .reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=dakar,
        scale=100,
        maxPixels=1e8,
        bestEffort=True
    ).getInfo()

area_km2 = round(flood_area.get('VV', 0) / 1e6, 2)
print(f"Surface inondée (saison 2020) : {area_km2} km²")

# Période sèche
s1_dry = (ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterBounds(dakar)
    .filterDate('2020-01-01', '2020-02-28')
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    .filter(ee.Filter.eq('orbitProperties_pass', 'ASCENDING'))
    .select('VV')
    .mean())

dry_mask = s1_dry.lt(WATER_THRESHOLD)
dry_area = dry_mask.multiply(ee.Image.pixelArea()) \
    .reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=dakar,
        scale=100,
        maxPixels=1e8,
        bestEffort=True
    ).getInfo()

dry_km2 = round(dry_area.get('VV', 0) / 1e6, 2)
print(f"Surface eau période sèche : {dry_km2} km²")
print(f"Delta inondation : {round(area_km2 - dry_km2, 2)} km²")

EEException: Image.select: Band pattern 'permanent_water' did not match any bands. Available bands: [occurrence, change_abs, change_norm, seasonality, recurrence, transition, max_extent]

In [30]:
# Diagnostic : comparer bbox grande vs urbaine + tester seuils
# (utiliser données déjà chargées dans le kernel)

print("="*70)
print("DIAGNOSTIC ZONES ET SEUILS")
print("="*70)

# ZONE 1: Dakar entière (ancienne - trop grande)
dakar_large = ee.Geometry.Rectangle([-17.6, 14.6, -17.1, 14.95])

# ZONE 2: Dakar urbaine réduite
dakar_urban = ee.Geometry.Rectangle([-17.52, 14.65, -17.25, 14.85])

print("\n📊 ANALYSE DES PROBLÈMES:")
print("-" * 70)

print("\n1️⃣  LA BBOX EST TROP GRANDE")
print("    Dakar large: [-17.6, 14.6, -17.1, 14.95]")
print("    → Surface: ~75 km (E-O) x 35 km (N-S) = ~1395 km²")
print("    ⚠️  Inclut l'OCÉAN ATLANTIQUE à l'ouest!")
print("    → Solution: Réduire à zone urbaine uniquement")
print("    Dakar urbain: [-17.52, 14.65, -17.25, 14.85]")
print("    → Surface: ~30 km (E-O) x 20 km (N-S) = ~200-250 km² attendus")

print("\n2️⃣  SEUIL -15 dB TROP PERMISSIF")
print("    Caractéristiques backscatter SAR:")
print("    • Eau libre:      -22 à -20 dB (très bas)")
print("    • Béton urbain:   -15 à -10 dB (plus haut)")
print("    • Routes humides: -18 à -16 dB (intermédiaire)")
print("    ⚠️  Seuil -15 dB capture aussi le béton/infrastructure urbaine")
print("    → Solution: Utiliser seuil STRICT: -20 dB")

print("\n3️⃣  DONNÉES SAR LIMITÉES")
print("    Situation en août 2020 à Dakar:")
print("    • Beaucoup d'images Sentinel-1 région: polarisations HH/HV seulement")
print("    • Pas de VV disponible partout à cette date")
print("    • Solution: Utiliser VV quand disponible, adapter si besoin")

print("\n" + "="*70)
print("✅ SOLUTION RECOMMANDÉE:")
print("="*70)
print("✓ BBOX:     Zone urbaine: [-17.52, 14.65, -17.25, 14.85]")
print("✓ SEUIL:    -20 dB (au lieu de -15 dB)")
print("✓ MASQUE:   Exclure eau permanente (JRC GSW occurrence > 80%)")
print("✓ BANDE:    VV si disponible, VH en fallback")
print("="*70)
print("\n💡 RÉSULTAT ATTENDU:")
print("   Delta inondation positif = plus d'eau en saison pluies ✓")
print("="*70)

DIAGNOSTIC ZONES ET SEUILS

📊 ANALYSE DES PROBLÈMES:
----------------------------------------------------------------------

1️⃣  LA BBOX EST TROP GRANDE
    Dakar large: [-17.6, 14.6, -17.1, 14.95]
    → Surface: ~75 km (E-O) x 35 km (N-S) = ~1395 km²
    ⚠️  Inclut l'OCÉAN ATLANTIQUE à l'ouest!
    → Solution: Réduire à zone urbaine uniquement
    Dakar urbain: [-17.52, 14.65, -17.25, 14.85]
    → Surface: ~30 km (E-O) x 20 km (N-S) = ~200-250 km² attendus

2️⃣  SEUIL -15 dB TROP PERMISSIF
    Caractéristiques backscatter SAR:
    • Eau libre:      -22 à -20 dB (très bas)
    • Béton urbain:   -15 à -10 dB (plus haut)
    • Routes humides: -18 à -16 dB (intermédiaire)
    ⚠️  Seuil -15 dB capture aussi le béton/infrastructure urbaine
    → Solution: Utiliser seuil STRICT: -20 dB

3️⃣  DONNÉES SAR LIMITÉES
    Situation en août 2020 à Dakar:
    • Beaucoup d'images Sentinel-1 région: polarisations HH/HV seulement
    • Pas de VV disponible partout à cette date
    • Solution: Utiliser

In [31]:
# Application des corrections recommandées

print("\n" + "="*70)
print("CALCUL CORRIGÉ: Seuil -20 dB + Zone urbaine réduite")
print("="*70 + "\n")

# === CORRECTIONS ===
WATER_THRESHOLD_STRICT = -20  # était -15, trop permissif
dakar_urban = ee.Geometry.Rectangle([-17.52, 14.65, -17.25, 14.85])

# Charger JRC GSW pour masquer eau permanente
jrc = ee.Image('JRC/GSW1_4/GlobalSurfaceWater')
permanent_water = jrc.select('occurrence').gt(80)

# === SAISON PLUIES (août-septembre 2020) ===
s1_flood_event_corrected = (ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterBounds(dakar_urban)
    .filterDate('2020-08-01', '2020-09-30')
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    .filter(ee.Filter.eq('orbitProperties_pass', 'ASCENDING'))
    .select('VV')
    .mean())

water_mask_corrected = s1_flood_event_corrected.lt(WATER_THRESHOLD_STRICT)
water_mask_corrected = water_mask_corrected.where(permanent_water, 0)  # exclure océan

flood_area_corrected = water_mask_corrected.multiply(ee.Image.pixelArea()) \
    .reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=dakar_urban,
        scale=100,
        maxPixels=1e8,
        bestEffort=True
    ).getInfo()

area_km2_corrected = round(flood_area_corrected.get('VV', 0) / 1e6, 2)

# === PÉRIODE SÈCHE (janvier-février 2020) ===
s1_dry_corrected = (ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterBounds(dakar_urban)
    .filterDate('2020-01-01', '2020-02-28')
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    .filter(ee.Filter.eq('orbitProperties_pass', 'ASCENDING'))
    .select('VV')
    .mean())

dry_mask_corrected = s1_dry_corrected.lt(WATER_THRESHOLD_STRICT)
dry_mask_corrected = dry_mask_corrected.where(permanent_water, 0)

dry_area_corrected = dry_mask_corrected.multiply(ee.Image.pixelArea()) \
    .reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=dakar_urban,
        scale=100,
        maxPixels=1e8,
        bestEffort=True
    ).getInfo()

dry_km2_corrected = round(dry_area_corrected.get('VV', 0) / 1e6, 2)

# === AFFICHAGE DES RÉSULTATS ===
print("PARAMÈTRES:")
print(f"  Seuil SAR:        {WATER_THRESHOLD_STRICT} dB (strict)")
print(f"  Zone:             Urbaine réduite [-17.52, 14.65, -17.25, 14.85]")
print(f"  Masque:           JRC GSW occurrence > 80%")
print(f"  Résolution:       100 m")
print()
print("RÉSULTATS:")
print(f"  Surface inondée (saison pluies 2020):    {area_km2_corrected:7.2f} km²")
print(f"  Surface eau (période sèche 2020):        {dry_km2_corrected:7.2f} km²")
print()
delta_corrected = round(area_km2_corrected - dry_km2_corrected, 2)
print(f"  ✅ DELTA INONDATION:                      {delta_corrected:7.2f} km²")
print()
if delta_corrected > 0:
    print(f"  ✓ POSITIF: Plus d'eau en saison pluies ✓")
else:
    print(f"  ⚠️  NÉGATIF: Toujours plus d'eau en saison sèche")
print("="*70)


CALCUL CORRIGÉ: Seuil -20 dB + Zone urbaine réduite

PARAMÈTRES:
  Seuil SAR:        -20 dB (strict)
  Zone:             Urbaine réduite [-17.52, 14.65, -17.25, 14.85]
  Masque:           JRC GSW occurrence > 80%
  Résolution:       100 m

RÉSULTATS:
  Surface inondée (saison pluies 2020):       1.16 km²
  Surface eau (période sèche 2020):           1.12 km²

  ✅ DELTA INONDATION:                         0.04 km²

  ✓ POSITIF: Plus d'eau en saison pluies ✓


In [32]:
print("Distribution des scores terrain (200 points) :")
print(df_grid['terrain_risk'].describe().round(2))
print()

# Répartition par catégorie de risque
bins   = [0, 25, 50, 75, 90, 100]
labels = ['Faible', 'Modéré', 'Élevé', 'Très élevé', 'Extrême']
df_grid['risk_class'] = pd.cut(df_grid['terrain_risk'], bins=bins, labels=labels)

print("Répartition par classe de risque :")
print(df_grid['risk_class'].value_counts().sort_index().to_string())
print()

# Combien de points à risque extrême (100/100) ?
extreme = (df_grid['terrain_risk'] == 100).sum()
print(f"Points à risque extrême (100/100) : {extreme}/200 = {extreme/2:.0f}% de la zone")

Distribution des scores terrain (200 points) :


NameError: name 'df_grid' is not defined

## 📋 Résumé et Leçons Apprises

### Problèmes Identifiés

| Problème | Cause | Impact | Solution |
|----------|-------|--------|----------|
| **Delta négatif (-77.75 km²)** | Seuil trop permissif (-15 dB) | Capture béton + océan en période sèche | Seuil strict -20 dB |
| **Bbox trop grande (1395 km²)** | Inclut océan Atlantique | Surface d'eau de base énorme | Zone urbaine réduite |
| **Faux positifs urbains** | Béton/métal ≈ eau en SAR | Détections incorrectes | Masque JRC GSW |

### Résultats

- **Avant corrections**: Delta = -77.75 km² (❌ négatif = erreur)
- **Après corrections**: Delta = +0.04 km² (✅ positif mais petit)

### Interprétations Possibles

1. **Zone urbaine trop petite** → peu de surface inondable naturelle
2. **Différence saisonnière faible** → janvier-février aussi humide qu'août-septembre
3. **Zones inondables concentrées** → quartiers spécifiques (Pikine, Guédiawaye) à zoomer davantage

### Prochaines Étapes

- ✅ Utiliser seuil -20 dB (non -15)
- ✅ Réduire bbox à zone urbaine (non côtière)
- ✅ Masquer eau permanente (JRC GSW)
- 🔍 **À tester**: Zoomer sur zones inondables connues (Pikine, Guédiawaye)
- 🔍 **À tester**: Comparer avec données CHIRPS (précipitation J-7)
- 🔍 **À tester**: Utiliser indices hydrologie alternatifs (NDVI, indices d'humidité)

## Test the API

Test the API by printing the elevation of Mount Everest.